# Notebook Overview — Basic Testing

## Purpose

This notebook performs **final independent evaluation** of a baseline classifier for the Digital Image Processing (DIP)–based AI image detection pipeline.

A **Multi-Layer Perceptron (MLP)** model is trained using normalized DIP feature vectors and evaluated on a **held-out test dataset**, providing an unbiased assessment of baseline model performance prior to further tuning.

---

## Inputs

Required inputs:

* Feature vector CSV files:
  * `metadata/vectors/<train_feature_vectors_filename>`
  * `metadata/vectors/<test_feature_vectors_filename>`

Shared configuration:

* `src/project_config.py`

---

## Execution Model

* Training and test feature vector datasets are loaded
* Metadata and feature columns are validated and separated
* Feature dimensionality and label consistency are verified
* Features are normalized using a `StandardScaler` fit on training data only
* A baseline MLP classifier is defined with fixed architecture and hyperparameters
* The model is trained on normalized training data
* Predictions and class probabilities are generated for the test dataset
* Evaluation metrics are computed:
  * Accuracy
  * Precision
  * Recall
  * F1 score
  * ROC AUC
* Confusion matrix and classification report are generated
* Results are summarized in tabular and visual formats
* Evaluation outputs and model configuration are saved
* Processing is deterministic and reproducible

---

## Outputs

**Baseline Test Results**  
`metadata/results/<basic_testing_results_filename>`

**Baseline Model Configuration**  
`metadata/results/<baseline_model_config_filename>`

**Saved Scaler**  
`metadata/models/<scaler_filename>`

Each results file includes:

* Model metadata and configuration
* Performance metrics on the independent test set
* Dataset sizes and feature count
* Training diagnostics (iterations, final loss)

---

## Expected Behavior

* Training and test datasets load successfully with valid structure
* Feature dimensionality remains consistent (26 features)
* No missing values are present in the data
* Feature normalization is correctly applied (train mean ≈ 0, std ≈ 1)
* Baseline model trains successfully within defined iterations
* Model produces valid predictions and probability outputs
* Evaluation metrics reflect unbiased test-set performance
* Output files are saved and verified without error
* Results provide a baseline reference for subsequent model improvements

---
---

### 🔷 Step 1 — Startup: Environment and Verification

- Import required libraries for model evaluation and preprocessing
- Configure notebook display behavior using `VERBOSE`
- Optionally suppress warnings for cleaner output
- Clone the GitHub repository if needed
- Add the project `src` directory to the Python path
- Import shared configuration values from `project_config.py`
- Define input paths for training and test feature vectors
- Define output paths for evaluation results, model configuration, and scaler
- Create required output directories if they do not exist
- Verify that required input files are available
- Optionally display configuration paths when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 1: Startup — Environment and Verification
# ============================================

# -------------------------------------------------
# Import required libraries
# -------------------------------------------------
import os
import sys
import warnings
from pathlib import Path

import pandas as pd

from sklearn.preprocessing import StandardScaler
import joblib

# -------------------------------------------------
# Notebook display control
# -------------------------------------------------
VERBOSE = True   # Set to False to reduce detailed output

# -------------------------------------------------
# Suppress warnings for cleaner output (optional)
# -------------------------------------------------
if not VERBOSE:
    warnings.filterwarnings("ignore")

# -------------------------------------------------
# Clone GitHub repository if needed
# -------------------------------------------------
REPO_URL = "https://github.com/pgailinas/dip-ai-image-detection.git"
REPO_DIR = Path("/content/dip-ai-image-detection")

if not REPO_DIR.exists():
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# -------------------------------------------------
# Add src directory to Python path
# -------------------------------------------------
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# -------------------------------------------------
# Import shared project configuration
# -------------------------------------------------
from project_config import *

# -------------------------------------------------
# Convert configured paths to Path objects
# -------------------------------------------------
train_feature_path = Path(TRAIN_FEATURE_VECTOR_PATH)
test_feature_path = Path(TEST_FEATURE_VECTOR_PATH)

results_dir = Path(RESULTS_METADATA_DIR)
models_dir = Path(MODELS_METADATA_DIR)

basic_testing_results_path = Path(BASIC_TESTING_RESULTS_PATH)
baseline_model_config_path = Path(BASELINE_MODEL_CONFIG_PATH)
scaler_path = Path(SCALER_PATH)

# -------------------------------------------------
# Ensure required output directories exist
# -------------------------------------------------
results_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Verify required input files
# -------------------------------------------------
print("Verifying required input files...\n")

required_files = {
    "Training feature vectors": train_feature_path,
    "Test feature vectors": test_feature_path,
}

missing_files = [
    f"{description}: {path}"
    for description, path in required_files.items()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing required input file(s):\n" + "\n".join(missing_files)
    )

print("All required input files are present.")

# -------------------------------------------------
# Optional verbose display of configuration paths
# -------------------------------------------------
if VERBOSE:
    print(f"\nTraining features: {train_feature_path}")
    print(f"Test features:     {test_feature_path}")
    print(f"Results output:    {basic_testing_results_path}")
    print(f"Model config:      {baseline_model_config_path}")
    print(f"Scaler output:     {scaler_path}")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nStartup complete.")



### 🔷 Step 2 — Load Feature Vector Data

- Load training and test feature vector datasets from CSV files
- Confirm successful data loading and display dataset shapes
- Verify required metadata columns are present
- Validate subset labels (`train` and `test`)
- Ensure no missing values exist in either dataset
- Display class distribution for both training and test sets
- Optionally preview sample rows when `VERBOSE=True`
- Prepare datasets for normalization and evaluation steps

---

In [ ]:
# ============================================
# Step 2: Load Feature Vector Data
# ============================================

# -------------------------------------------------
# Load training and test feature vector datasets
# -------------------------------------------------
train_df = pd.read_csv(train_feature_path)
test_df  = pd.read_csv(test_feature_path)

print("Feature vector files loaded successfully.\n")

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape:     {test_df.shape}")

# -------------------------------------------------
# Verify required metadata columns are present
# -------------------------------------------------
required_columns = ["filename", "class_label", "source_dataset", "subset"]

missing_train = [col for col in required_columns if col not in train_df.columns]
missing_test  = [col for col in required_columns if col not in test_df.columns]

if missing_train:
    raise ValueError(f"Training file missing required columns: {missing_train}")

if missing_test:
    raise ValueError(f"Test file missing required columns: {missing_test}")

print("\nRequired metadata columns verified.")

# -------------------------------------------------
# Verify subset labels are correct
# -------------------------------------------------
train_subsets = train_df["subset"].unique()
test_subsets  = test_df["subset"].unique()

if list(train_subsets) != ["train"]:
    raise ValueError(f"Unexpected subset values in training data: {train_subsets}")

if list(test_subsets) != ["test"]:
    raise ValueError(f"Unexpected subset values in test data: {test_subsets}")

print("Subset labels verified.")

# -------------------------------------------------
# Check for missing values
# -------------------------------------------------
if train_df.isnull().values.any():
    raise ValueError("Training data contains missing values.")

if test_df.isnull().values.any():
    raise ValueError("Test data contains missing values.")

print("No missing values detected.")

# -------------------------------------------------
# Display class distribution
# -------------------------------------------------
print("\nTraining class distribution:")
print(train_df["class_label"].value_counts())

print("\nTest class distribution:")
print(test_df["class_label"].value_counts())

# -------------------------------------------------
# Optional verbose preview of data
# -------------------------------------------------
if VERBOSE:
    print("\nSample training rows:")
    display(train_df.head())

    print("\nSample test rows:")
    display(test_df.head())



### 🔷 Step 3 — Prepare Features and Labels

- Identify feature columns by excluding metadata columns
- Validate total number of features against expected count
- Construct feature matrices `X_train` and `X_test`
- Construct label vectors `y_train` and `y_test`
- Verify feature dimensionality consistency between train and test sets
- Validate class labels against expected label set
- Optionally display sample feature vector when `VERBOSE=True`
- Prepare data for normalization and model evaluation

---

In [ ]:
# ============================================
# Step 3: Prepare Features and Labels
# ============================================

# -------------------------------------------------
# Use configuration-defined metadata columns
# -------------------------------------------------
metadata_columns = METADATA_COLUMNS

# -------------------------------------------------
# Identify feature columns
# -------------------------------------------------
feature_columns = [col for col in train_df.columns if col not in metadata_columns]

print(f"Number of feature columns: {len(feature_columns)}")
print(f"Expected number of features: {NUM_FEATURES}")

# -------------------------------------------------
# Validate feature count
# -------------------------------------------------
if len(feature_columns) != NUM_FEATURES:
    raise ValueError(
        f"Feature count mismatch. Expected {NUM_FEATURES}, found {len(feature_columns)}"
    )

# -------------------------------------------------
# Create feature matrices for training and test sets
# -------------------------------------------------
X_train = train_df[feature_columns].values
X_test  = test_df[feature_columns].values

# -------------------------------------------------
# Create label vectors for training and test sets
# -------------------------------------------------
y_train = train_df["class_label"].values
y_test  = test_df["class_label"].values

# -------------------------------------------------
# Verify feature dimensionality consistency
# -------------------------------------------------
if X_train.shape[1] != X_test.shape[1]:
    raise ValueError(
        f"Feature dimension mismatch: train={X_train.shape[1]}, test={X_test.shape[1]}"
    )

print("\nFeature matrices created.")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

# -------------------------------------------------
# Validate class labels
# -------------------------------------------------
unique_labels = sorted(set(y_train) | set(y_test))

print(f"\nUnique class labels: {unique_labels}")

if set(unique_labels) != set(VALID_LABELS):
    raise ValueError(
        f"Unexpected class labels. Expected {VALID_LABELS}, found {unique_labels}"
    )

print("Class labels verified.")

# -------------------------------------------------
# Optional verbose display
# -------------------------------------------------
if VERBOSE:
    print("\nSample feature vector (first row):")
    print(X_train[0])



### 🔷 Step 4 — Normalize Features

- Initialize `StandardScaler` for feature normalization
- Fit scaler using **training data only** to prevent data leakage
- Apply learned transformation to both training and test feature matrices
- Verify normalized feature shapes
- Save fitted scaler to disk for reuse in future inference
- Perform sanity check:
  - Training feature means ≈ 0
  - Training feature standard deviations ≈ 1
- Optionally display normalization statistics when `VERBOSE=True`
- Ensure consistent preprocessing for model evaluation

---

In [ ]:
# ============================================
# Step 4: Normalize Features
# ============================================

# -------------------------------------------------
# Define scaler output path
# -------------------------------------------------
scaler_file = scaler_path

# -------------------------------------------------
# Fit scaler on training data and transform datasets
# -------------------------------------------------
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)  # Fit on training data only
X_test_scaled  = scaler.transform(X_test)       # Apply same transformation to test data

print("Feature normalization complete.\n")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape:  {X_test_scaled.shape}")

# -------------------------------------------------
# Save fitted scaler to disk
# -------------------------------------------------
joblib.dump(scaler, scaler_file)

if not scaler_file.exists():
    raise FileNotFoundError(f"Scaler file was not created: {scaler_file}")

print(f"Scaler saved to: {scaler_file}")

# -------------------------------------------------
# Sanity check on normalized training data
# -------------------------------------------------
train_mean = X_train_scaled.mean(axis=0)
train_std  = X_train_scaled.std(axis=0)

if VERBOSE:
    print("\nNormalization sanity check:")
    print(f"Mean of first 5 scaled features: {train_mean[:5]}")
    print(f"Std of first 5 scaled features:  {train_std[:5]}")


### 🔷 Step 5 — Define Baseline Classifier

- Define baseline **Multi-Layer Perceptron (MLP)** classifier
- Configure architecture and hyperparameters:
  - Hidden layers: (64, 32)
  - Activation: ReLU
  - Optimizer: Adam
  - Regularization (alpha): 0.0001
  - Batch size: 32
  - Learning rate: 0.001
  - Maximum iterations: 300
- Disable early stopping for consistent training behavior
- Confirm model type and configuration
- Optionally display full parameter settings when `VERBOSE=True`
- Prepare baseline model for training and evaluation

---

In [ ]:
# ============================================
# Step 5: Define Baseline Classifier
# ============================================

from sklearn.neural_network import MLPClassifier

# -------------------------------------------------
# Define baseline Multi-Layer Perceptron (MLP) model
# -------------------------------------------------
baseline_mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=32,
    learning_rate_init=0.001,
    max_iter=300,
    random_state=RANDOM_SEED,
    early_stopping=False,
)

print("Baseline classifier defined.\n")

print("Model type:")
print(type(baseline_mlp).__name__)

# -------------------------------------------------
# Optional verbose display of model configuration
# -------------------------------------------------
if VERBOSE:
    print("\nBaseline MLP configuration:")
    print(f"hidden_layer_sizes : {baseline_mlp.hidden_layer_sizes}")
    print(f"activation         : {baseline_mlp.activation}")
    print(f"solver             : {baseline_mlp.solver}")
    print(f"alpha              : {baseline_mlp.alpha}")
    print(f"batch_size         : {baseline_mlp.batch_size}")
    print(f"learning_rate_init : {baseline_mlp.learning_rate_init}")
    print(f"max_iter           : {baseline_mlp.max_iter}")
    print(f"random_state       : {baseline_mlp.random_state}")
    print(f"early_stopping     : {baseline_mlp.early_stopping}")



### 🔷 Step 6 — Train Baseline Model

- Train the baseline MLP classifier using normalized training data
- Measure and report total training time
- Capture and monitor convergence warnings during optimization
- Display training diagnostics:
  - Number of iterations completed
  - Final training loss
- Detect whether the model fully converged
- Provide warning if convergence was not achieved
- Confirm successful completion of model training

---

In [ ]:
# ============================================
# Step 6: Train Baseline Model
# ============================================

import time
import warnings
from sklearn.exceptions import ConvergenceWarning

# -------------------------------------------------
# Train baseline MLP model
# -------------------------------------------------
print("Training baseline MLP model...\n")

start_time = time.time()

# Capture convergence warnings during training
with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always", ConvergenceWarning)
    baseline_mlp.fit(X_train_scaled, y_train)

end_time = time.time()
training_time = end_time - start_time

print("Training complete.\n")
print(f"Training time: {training_time:.2f} seconds")

# -------------------------------------------------
# Display training diagnostics (optional)
# -------------------------------------------------
if VERBOSE:
    print(f"\nNumber of iterations completed: {baseline_mlp.n_iter_}")
    print(f"Final training loss: {baseline_mlp.loss_:.6f}")

# -------------------------------------------------
# Check for convergence warnings
# -------------------------------------------------
convergence_warnings = [
    w for w in caught_warnings
    if issubclass(w.category, ConvergenceWarning)
]

if convergence_warnings:
    print("\nWARNING: Convergence warning detected.")
    print("The optimizer did not fully converge within the allowed iterations.")
else:
    print("\nModel converged without warnings.")



### 🔷 Step 7 — Evaluate on Test Set

- Generate predictions using the trained baseline model
- Compute class probabilities for ROC AUC evaluation
- Convert ground truth labels to binary format for ROC calculation
- Compute evaluation metrics:
  - Accuracy
  - Precision
  - Recall
  - F1 Score
  - ROC AUC
- Generate confusion matrix (real vs AI classification)
- Optionally display full classification report when `VERBOSE=True`
- Provide final performance assessment on the held-out test dataset

---

In [ ]:
# ============================================
# Step 7: Evaluate on Test Set
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

# -------------------------------------------------
# Generate predictions from trained model
# -------------------------------------------------
y_pred = baseline_mlp.predict(X_test_scaled)

# Identify index of AI class for probability extraction
ai_index = list(baseline_mlp.classes_).index(AI_LABEL)

# Compute predicted probabilities for AI class
y_prob = baseline_mlp.predict_proba(X_test_scaled)[:, ai_index]

# -------------------------------------------------
# Convert labels to binary format for ROC AUC
# -------------------------------------------------
y_test_binary = (y_test == AI_LABEL).astype(int)

# -------------------------------------------------
# Compute evaluation metrics
# -------------------------------------------------
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=AI_LABEL)
recall    = recall_score(y_test, y_pred, pos_label=AI_LABEL)
f1        = f1_score(y_test, y_pred, pos_label=AI_LABEL)
roc_auc   = roc_auc_score(y_test_binary, y_prob)

# Compute confusion matrix (ordered: real, ai)
cm = confusion_matrix(y_test, y_pred, labels=[REAL_LABEL, AI_LABEL])

# -------------------------------------------------
# Display evaluation results
# -------------------------------------------------
print("Baseline test-set evaluation complete.\n")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix (rows=true, cols=pred):")
print(cm)

# -------------------------------------------------
# Optional detailed classification report
# -------------------------------------------------
if VERBOSE:
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=4))



### 🔷 Step 8 — Display Results

- Create a compact summary table of key performance metrics:
  - Accuracy
  - Precision
  - Recall
  - F1 Score
  - ROC AUC
- Display results in a structured tabular format
- Optionally visualize model performance:
  - Confusion matrix (classification breakdown)
  - ROC curve (classifier discrimination ability)
- Use visualizations to support interpretation of model performance
- Provide clear summary of baseline classifier effectiveness

---

In [ ]:
# ============================================
# Step 8: Display Results
# ============================================

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
import pandas as pd

# -------------------------------------------------
# Create compact results summary table
# -------------------------------------------------
results_summary_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC AUC"],
    "Value": [accuracy, precision, recall, f1, roc_auc]
})

print("Baseline performance summary:")
display(results_summary_df)

# -------------------------------------------------
# Optional visualization of results
# -------------------------------------------------
if VERBOSE:
    # -------------------------------------------------
    # Display confusion matrix
    # -------------------------------------------------
    disp_cm = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[REAL_LABEL, AI_LABEL]
    )

    plt.figure(figsize=(6, 6))
    disp_cm.plot(cmap="Blues", values_format="d")
    plt.title("Baseline MLP Confusion Matrix")
    plt.grid(False)
    plt.show()

    # -------------------------------------------------
    # Display ROC curve
    # -------------------------------------------------
    RocCurveDisplay.from_predictions(
        y_test_binary,
        y_prob,
        name="Baseline MLP"
    )

    plt.title("Baseline MLP ROC Curve")
    plt.grid(True)
    plt.show()



### 🔷 Step 9 — Save Results

- Create structured results dictionary containing:
  - Performance metrics (accuracy, precision, recall, F1, ROC AUC)
  - Dataset sizes (train/test)
  - Feature count
  - Training diagnostics (iterations, final loss)
- Save results to CSV file for reporting and analysis
- Verify successful file creation
- Save model configuration (hyperparameters) to JSON file
- Ensure reproducibility of baseline model setup
- Confirm successful persistence of results and configuration

---

In [ ]:
# ============================================
# Step 9: Save Results
# ============================================

import pandas as pd
import json

# -------------------------------------------------
# Create results dictionary
# -------------------------------------------------
results_dict = {
    "model": "MLP (64, 32)",
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "roc_auc": roc_auc,
    "train_samples": len(X_train),
    "test_samples": len(X_test),
    "num_features": X_train.shape[1],
    "training_iterations": baseline_mlp.n_iter_,
    "final_loss": baseline_mlp.loss_,
}

# -------------------------------------------------
# Save results to CSV file
# -------------------------------------------------
results_df = pd.DataFrame([results_dict])
results_df.to_csv(basic_testing_results_path, index=False)

# -------------------------------------------------
# Verify results file creation
# -------------------------------------------------
if not basic_testing_results_path.exists():
    raise FileNotFoundError(
        f"Results file was not created: {basic_testing_results_path}"
    )

print(f"Results saved to: {basic_testing_results_path}")

# -------------------------------------------------
# Save model configuration to JSON file
# -------------------------------------------------
model_config = {
    "model_type": "MLPClassifier",
    "hidden_layer_sizes": baseline_mlp.hidden_layer_sizes,
    "activation": baseline_mlp.activation,
    "solver": baseline_mlp.solver,
    "alpha": baseline_mlp.alpha,
    "batch_size": baseline_mlp.batch_size,
    "learning_rate_init": baseline_mlp.learning_rate_init,
    "max_iter": baseline_mlp.max_iter,
    "random_state": baseline_mlp.random_state,
}

with open(baseline_model_config_path, "w") as f:
    json.dump(model_config, f, indent=4)

print(f"Model configuration saved to: {baseline_model_config_path}")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nCell complete.")



### 🔷 Step 10 — Final Summary and Next Step

- Display final baseline testing summary
- Report trained model architecture and training diagnostics
- Summarize independent test-set performance metrics
- Display confusion matrix interpretation by class
- Provide next-step guidance for controlled fine-tuning
- Render formatted completion message

---

In [ ]:
# ============================================
# Step 10: Final Summary and Next Step
# ============================================

from IPython.display import display, HTML

# -------------------------------------------------
# Display final baseline testing summary
# -------------------------------------------------
print("Basic baseline testing completed successfully.\n")

print("Baseline Model Summary:")
print(f"Model type         : {type(baseline_mlp).__name__}")
print(f"Architecture       : {baseline_mlp.hidden_layer_sizes}")
print(f"Training samples   : {len(X_train)}")
print(f"Test samples       : {len(X_test)}")
print(f"Feature count      : {X_train.shape[1]}")
print(f"Training iterations: {baseline_mlp.n_iter_}")
print(f"Final training loss: {baseline_mlp.loss_:.6f}")

# -------------------------------------------------
# Display final test metrics
# -------------------------------------------------
print("\nBaseline Test Metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC AUC  : {roc_auc:.4f}")

# -------------------------------------------------
# Display confusion matrix summary
# -------------------------------------------------
print("\nConfusion Matrix Summary:")
print(f"Real ({REAL_LABEL}): {cm[0,0]} correct, {cm[0,1]} misclassified")
print(f"AI   ({AI_LABEL}): {cm[1,1]} correct, {cm[1,0]} misclassified")
print()

# -------------------------------------------------
# Define next-step guidance message
# -------------------------------------------------
message = """
<b>Baseline Testing Complete:</b><br>
The baseline MLP model has been trained and evaluated on the independent test set.<br><br>
Proceed to <b>11_Basic_Fine-Tuning.ipynb</b> to explore controlled improvement through limited hyperparameter tuning.
"""

# -------------------------------------------------
# Render formatted completion message
# -------------------------------------------------
display(HTML(f"""
<div style="
    padding: 15px;
    border: 2px solid #4CAF50;
    background-color: #E8F5E9;
    border-radius: 8px;
    font-size: 16px;
">
{message}
</div>
"""))

